# Phase 4: does it generalize? The zero-shot test

The v1 general tracker plays motions it never trained on, next to plain replay (no policy) and, for
the course tricks, the specialist trained on exactly that trick.

| set | what | from |
|---|---|---|
| **4a** | the benchmark's keyframe motions (12 requests × 2 repeats); 10 have precision checks | `test_v1` (built from the benchmark's results.pkl) |
| **4b** | ~30 new requests, written now, unlike any training request | this notebook (Colab AI) |
| **4c** | the held-out course tricks: sneeze, push_up, stand_sit_shake_sit_stand | `test_v1` (+ their specialists' `policy.json`) |

Scores are `play_motion`'s (worst position / tilt error, HOLDS UP ≤ 3 cm and 10°, FALLS > 10 cm or 45°),
from `kflib/track.py`. That's the same scoring loop, plus the trajectory the robot actually followed, so the
benchmark's precision checks can be run on what the robot *did*. **Noise:** the simulation isn't
bit-for-bit repeatable. Two runs of the same plain replay differed by ~0.3 cm and up to ~5° on a
wobbly motion, so a motion right at the 10° line can flip verdicts. Read counts of ±1-2 as noise.

Needs a GPU runtime (G4 or A100) and `test_v1` uploaded to `MyDrive/pupper_general_tracker/datasets/`.
**Nothing here moves the robot.**

In [ ]:
# @title Setup (about 2 minutes)
import os, sys, subprocess
COMMIT = "aa475e9a393a209b29fd5de7e0ee10f3de643230"
BRANCH = "final-project-kflib"  # @param {type:"string"}
RUN_NAME = "pgt_v1"  # @param {type:"string"}
print(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout or "NO GPU")
if not os.path.exists("/content/pupper-mjlab"):
  !git clone -q https://github.com/cs123-stanford/pupper-mjlab /content/pupper-mjlab
!git -C /content/pupper-mjlab checkout -q {COMMIT}
!pip install -q -e /content/pupper-mjlab
sys.path.insert(0, "/content/pupper-mjlab/src")
if not os.path.exists("/content/gemini_pupper"):
  !git clone -q -b {BRANCH} https://github.com/MlsBli/gemini_pupper /content/gemini_pupper
!git -C /content/gemini_pupper pull -q
sys.path.insert(0, "/content/gemini_pupper/final_project")
os.chdir("/content/gemini_pupper/final_project")

from google.colab import drive
drive.mount("/content/drive")
from pathlib import Path
ROOT = Path("/content/drive/MyDrive/pupper_general_tracker")
DATASET, TEST, RUNS = ROOT / "datasets" / "v1", ROOT / "datasets" / "test_v1", ROOT / "runs"
from kflib.runs import latest_checkpoint
from kflib.testset import TestSet
ts = TestSet(TEST)
IT, CKPT = latest_checkpoint(RUNS, RUN_NAME)
V1_JSON = CKPT.parent / "policy.json"  # the trick runner's export of that same checkpoint
RESULTS = TEST / f"results_{RUN_NAME}_{IT + 1}"
WORKERS = min(8, os.cpu_count())
print(len(ts.rows()), "test motions | v1 =", CKPT.parent.name, CKPT.name, "| results ->", RESULTS)

## Self-checks first (~3 minutes)

1. **The JSON path is right.** The specialists only exist as `policy.json`, so they're run with
   pupper-mjlab's `json_forward`. v1's own `policy.json` is the same network as its checkpoint, so
   playing a motion both ways must agree (to within the noise).
2. **track.py matches play_motion** on v1: the same motions as `eval_<iteration>.csv` from Phase 3.

Don't trust the specialist numbers below unless check 1 passes.

In [ ]:
# @title Self-checks
import csv, json
from kflib.track import run_many
ev = list(csv.DictReader((RUNS / f"eval_{RUN_NAME}" / f"eval_{IT}.csv").open()))
pick = [r for r in ev if r["split"] == "train"][:2]
jobs = []
for r in pick:
  npz = str(DATASET / "motions" / f"{r['id']}.npz")
  jobs += [dict(id=r["id"], npz=npz, mode="checkpoint", policy=str(CKPT), label="ckpt"),
           dict(id=r["id"], npz=npz, mode="json", policy=str(V1_JSON), label="json")]
res = run_many(jobs, RESULTS / "selfcheck", workers=4)
for r in pick:
  a = next(x for x in res if x["id"] == r["id"] and x["label"] == "ckpt")
  b = next(x for x in res if x["id"] == r["id"] and x["label"] == "json")
  ok = abs(a["pos_cm"] - b["pos_cm"]) <= 0.5 and abs(a["tilt_deg"] - b["tilt_deg"]) <= 2.0
  print(f"{'PASS' if ok else 'FAIL'}  JSON = checkpoint on {r['id']}: {b['pos_cm']} vs {a['pos_cm']} cm, {b['tilt_deg']} vs {a['tilt_deg']} deg")
  ok = abs(a["pos_cm"] - float(r["pos_cm"])) <= 0.5 and abs(a["tilt_deg"] - float(r["tilt_deg"])) <= 2.0
  print(f"{'PASS' if ok else 'FAIL'}  track.py = play_motion on {r['id']}: {a['pos_cm']} vs {r['pos_cm']} cm, {a['tilt_deg']} vs {r['tilt_deg']} deg")

## 4b: new requests (~15-20 minutes)

Written now, after training: Colab AI writes ~40 requests, the filter drops anything like a training
request (or the held-out tricks, or the benchmark), and each survivor goes through the usual
keyframes → gate pipeline. The first 30 that pass become test motions.

In [ ]:
# @title New requests -> test motions
from kflib.llm import LLM
from kflib.testset import add_new_requests
llm = LLM("colab")
print(add_new_requests(ts, llm, TEST / "requests_pool_new.jsonl", DATASET / "requests_pool.jsonl", n=30))
for r in [r for r in ts.rows() if r["set"] == "4b"][::5]:
  print(f"  [{r['kind']}] {r['request']}")

## Score everything (~15 minutes)

Every test motion by plain replay and by v1, and the three course tricks also by their specialists.
Results are saved per motion, so stopping and rerunning only does what's missing.

In [ ]:
# @title Score
from collections import Counter
jobs = []
for r in ts.rows():
  base = dict(id=r["id"], set=r["set"], source=r["source"], npz=str(ts.motions / f"{r['id']}.npz"))
  jobs += [dict(base, mode="plain", label="plain"), dict(base, mode="checkpoint", policy=str(CKPT), label="v1")]
  if r.get("specialist"):
    jobs.append(dict(base, mode="json", policy=str(TEST / r["specialist"]), label="specialist"))
print(len(jobs), "runs:", dict(Counter(j["label"] for j in jobs)))
rows = run_many(jobs, RESULTS, workers=WORKERS)

In [ ]:
# @title The zero-shot table, precision, and the figure
from IPython.display import Image, display
from kflib.zeroshot import figure, precision, table
print(f"{'set':4} {'':24} {'driver':11} {'n':>3}  stays up  HOLDS UP  median cm  deg   joints rad")
for t in table(rows):
  print(f"{t['set']:4} {t['name']:24} {t['driver']:11} {t['n']:3}  {t['stays_up']:>4}/{t['n']:<3}  {t['holds_up']:>4}/{t['n']:<3}"
        f"  {t['median_cm']:8}  {t['median_deg']:5}  {t['median_joint_rad']}")
print("\nprecision checks (benchmark's precise requests): the reference vs what the robot did under v1")
for p in precision(rows, ts, RESULTS):
  print(f"  {p['id']:18} {p['check']:10} reference {'ok ' if p['reference_ok'] else 'off'} {p['reference']}\n"
        f"  {'':29} tracked   {'ok ' if p['tracked_ok'] else 'off'} {p['tracked']}")
figure(rows, RESULTS / "zeroshot.png")
display(Image(str(RESULTS / "zeroshot.png")))

In [ ]:
# @title Where v1 doesn't hold up (watch these)
by_id = {r["id"]: r for r in ts.rows()}
plain = {r["id"]: r for r in rows if r["label"] == "plain"}
for r in sorted((r for r in rows if r["label"] == "v1" and r.get("verdict") != "HOLDS UP"), key=lambda r: -r["tilt_deg"]):
  p = plain.get(r["id"], {})
  print(f"{r['set']} {r['id']:22} v1 {r['verdict']} {r['pos_cm']} cm {r['tilt_deg']} deg | plain {p.get('verdict')} {p.get('pos_cm')} cm {p.get('tilt_deg')} deg"
        f"\n     {by_id[r['id']]['request'][:100]}")

In [ ]:
# @title (Optional) Watch one (open the share link; stop the cell when done)
WATCH = "course_sneeze"  # @param {type:"string"}
!python -m mjlab.tasks.tracking.config.pupper.play_motion {ts.motions / (WATCH + '.npz')} --checkpoint {CKPT} --viewer viser --share True --loops 2